In [ ]:
# Abrimos el archivo con datos limpios
import pandas as pd

df = pd.read_csv(r"C:\Users\damia\OneDrive\Desktop\SIMULACION EMPRESARIAL DATA ANALYST\Bank_Marketing_Cleaned-23_February.csv")

df.head()

In [ ]:
# Para saber si hay clientes que no han sido contactados

(df["campaign"] == 0).sum()

# **Calculos de KPI's**



In [ ]:
# Porcentaje de Conversion a Deposito

# Número clientes que suscribieron depósito
num_subscribers = (df["deposit"] == "yes").sum()

# Número total clientes contactados
total_clients = len(df)

# KPI Conversión
conversion_rate = (num_subscribers / total_clients) * 100

print(f"Porcentaje Conversión Depósito: {conversion_rate:.2f}%")

In [ ]:
# Para validar que no hayan valores vacios

df[df["deposit"]=="yes"]["duration"].isna().sum()

In [ ]:
# Promedio de Duracion de Llamadas de Suscriptores

# Filtrar suscriptores
subscribers = df[df["deposit"] == "yes"]

# Suma duración llamadas suscriptores
total_duration = subscribers["duration"].sum()

# Número suscriptores
num_subscribers = len(subscribers)

# KPI promedio duración
avg_duration_subscribers = (total_duration / num_subscribers)/60

print(f"Promedio duración llamadas suscriptores: {avg_duration_subscribers:.2f} minutos")

In [ ]:
# Porcentaje de Llamadas a Telefono o Movil

# Número llamadas realizadas por teléfono o móvil
calls_phone_mobile = df["contact"].isin(["telephone", "cellular"]).sum()

# Número total llamadas
total_calls = len(df)

# KPI porcentaje llamadas teléfono o móvil
percentage_calls = round(
    (calls_phone_mobile / total_calls) * 100,
    2
)

print(f"Porcentaje llamadas teléfono o móvil: {percentage_calls}%")

In [ ]:
# Promedio de Contactos Previos a Suscriptores

# Filtrar suscriptores
subscribers = df[df["deposit"] == "yes"]

# Suma contactos previos
total_previous_contacts = subscribers["previous"].sum()

# Número suscriptores
num_subscribers = len(subscribers)

# KPI promedio contactos previos
avg_previous_contacts = total_previous_contacts / num_subscribers

print(f"Promedio contactos previos suscriptores: {avg_previous_contacts:.2f}")

In [ ]:
# Mes con Mayor Tasa de Conversion

# Agrupar por mes
monthly_kpi = df.groupby("month").agg(

    # Número total clientes contactados en el mes
    total_clientes=("deposit", "count"),

    # Número suscriptores en el mes
    suscriptores=("deposit", lambda x: (x == "yes").sum())

)

# Calcular tasa conversión mensual con redondeo
monthly_kpi["conversion_rate"] = round(
    (monthly_kpi["suscriptores"] /
     monthly_kpi["total_clientes"]) * 100,
    2   # número de decimales
)

print("Conversión por mes:")
print(monthly_kpi)


# Mes con mayor conversión
best_month = monthly_kpi["conversion_rate"].idxmax()

best_value = monthly_kpi["conversion_rate"].max()

print(f"\nMes con mayor tasa de conversión: {best_month} ({best_value:.2f}%)")

In [ ]:
# %pip install plotly jinja2

In [33]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -------------------------------
# KPI 1 Conversion Rate
# -------------------------------

num_subscribers = (df["deposit"] == "yes").sum()
total_clients = len(df)
conversion_rate = (num_subscribers / total_clients) * 100


# -------------------------------
# KPI 2 Avg Duration Subscribers
# -------------------------------

subscribers = df[df["deposit"] == "yes"]

total_duration = subscribers["duration"].sum()
num_subscribers = len(subscribers)

avg_duration_subscribers = (total_duration / num_subscribers) / 60


# -------------------------------
# KPI 3 Phone/Mobile %
# -------------------------------

calls_phone_mobile = df["contact"].isin(["telephone", "cellular"]).sum()
total_calls = len(df)

percentage_calls = (calls_phone_mobile / total_calls) * 100


# -------------------------------
# KPI 4 Avg Previous Subscribers
# -------------------------------

total_previous_contacts = subscribers["previous"].sum()
avg_previous_contacts = total_previous_contacts / num_subscribers


# -------------------------------
# KPI 5 Best Month Conversion
# -------------------------------

monthly_kpi = df.groupby("month").agg(

    total_clientes=("deposit", "count"),

    suscriptores=("deposit", lambda x: (x == "yes").sum()),

    avg_duration=("duration",
    lambda x: x[df.loc[x.index,"deposit"]=="yes"].mean()/60),

    avg_previous=("previous",
    lambda x: x[df.loc[x.index,"deposit"]=="yes"].mean()),

    phone_mobile_pct=("contact",
    lambda x: x.isin(["telephone","cellular"]).mean()*100)

)

monthly_kpi["conversion_rate"] = (
monthly_kpi["suscriptores"] /
monthly_kpi["total_clientes"]
) * 100


best_month = monthly_kpi["conversion_rate"].idxmax()
best_value = monthly_kpi["conversion_rate"].max()


# -------------------------------
# Dashboard
# -------------------------------

fig = make_subplots(

rows=2,
cols=5,

specs=[
[{"type":"indicator"},{"type":"indicator"},{"type":"indicator"},{"type":"indicator"},{"type":"indicator"}],
[{"colspan":5},None,None,None,None]
]

)

# KPIs

fig.add_trace(go.Indicator(
mode="number",
value=conversion_rate,
title={"text":"Conversion Rate"},
number={"suffix":"%","font":{"size":50},"valueformat":".2f"}
),row=1,col=1)


fig.add_trace(go.Indicator(
mode="number",
value=avg_duration_subscribers,
title={"text":"Avg Duration (min)"},
number={"suffix":" min","font":{"size":50},"valueformat":".2f"}
),row=1,col=2)


fig.add_trace(go.Indicator(
mode="number",
value=avg_previous_contacts,
title={"text":"Avg Previous"},
number={"font":{"size":50},"valueformat":".2f"}
),row=1,col=3)


fig.add_trace(go.Indicator(
mode="number",
value=percentage_calls,
title={"text":"Phone/Mobile"},
number={"suffix":"%","font":{"size":50},"valueformat":".2f"}
),row=1,col=4)


fig.add_trace(go.Indicator(
mode="number",
value=best_value,
title={"text":f"Best Month ({best_month})"},
number={"suffix":"%","font":{"size":50},"valueformat":".2f"}
),row=1,col=5)


# -------------------------------
# Gráficos mensuales
# -------------------------------

fig.add_trace(go.Bar(
x=monthly_kpi.index,
y=monthly_kpi["conversion_rate"],
name="Conversion %",
visible=True
),row=2,col=1)


fig.add_trace(go.Bar(
x=monthly_kpi.index,
y=monthly_kpi["avg_duration"],
name="Avg Duration",
visible=False
),row=2,col=1)


fig.add_trace(go.Bar(
x=monthly_kpi.index,
y=monthly_kpi["avg_previous"],
name="Avg Previous",
visible=False
),row=2,col=1)


fig.add_trace(go.Bar(
x=monthly_kpi.index,
y=monthly_kpi["phone_mobile_pct"],
name="Phone/Mobile %",
visible=False
),row=2,col=1)


# -------------------------------
# Dropdown selector
# -------------------------------

buttons=[

dict(label="Conversion %",
method="update",
args=[{"visible":[True,True,True,True,True,True,False,False,False]}]),

dict(label="Avg Duration",
method="update",
args=[{"visible":[True,True,True,True,True,False,True,False,False]}]),

dict(label="Avg Previous",
method="update",
args=[{"visible":[True,True,True,True,True,False,False,True,False]}]),

dict(label="Phone/Mobile %",
method="update",
args=[{"visible":[True,True,True,True,True,False,False,False,True]}])

]

fig.update_layout(

updatemenus=[dict(
buttons=buttons,
direction="down",
x=0,
y=1.15
)],

height=650

)

fig.write_html("credit_risk_kpi_dashboard.html")